# LAB 01 — From Text Processing to Search: Experiments (Part D → J)

Corpus: `c4-train.00000-of-01024-30K.json.gz` (30,000 documents, C4) — đặt tại `lab01/data/`.

> **Các ô `✍️ TỰ VIẾT`** là phần phải tự trả lời (theo AI Usage Policy: giải thích kết quả,
> error analysis, reflection không được dùng AI). Code và số liệu bên dưới dùng để làm evidence.

In [1]:
import sys, time, json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from pipelines import (load_corpus, tokenize_a, tokenize_b, train_subword_tokenizer,
                       make_tokenize_c, pretokenize_c, make_vectorizer, TfidfSearch, preview)
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.width", 200, "display.max_colwidth", 80)
docs = load_corpus()
print("Loaded documents:", len(docs))

Loaded documents: 30000


## Part D — Experiment 1: Inspect the sparse representation
Pipeline: Raw documents → Tokenizer → CountVectorizer → TF → IDF → TF-IDF matrix
(tách riêng `CountVectorizer` và `TfidfTransformer` để thấy từng bước).

In [2]:
count_vec = CountVectorizer()                 # tokenizer mặc định: lowercase + \b\w\w+\b
C = count_vec.fit_transform(docs)             # count matrix
tfidf_tr = TfidfTransformer()                 # idf = ln((1+N)/(1+df)) + 1, L2 norm
X = tfidf_tr.fit_transform(C)
N, V = X.shape
print(f"Number of documents = {N}")
print(f"Vocabulary size     = {V}")
print(f"Matrix shape        = {X.shape}")
S = 1 - X.nnz / (N * V)
print(f"nnz(X)              = {X.nnz}")
print(f"Sparsity S          = {S:.6f}  ({S*100:.4f}% zero entries)")
print(f"Avg non-zero terms / doc = {X.nnz / N:.1f}  (so với V = {V})")

Number of documents = 30000
Vocabulary size     = 193540
Matrix shape        = (30000, 193540)
nnz(X)              = 4985822
Sparsity S          = 0.999141  (99.9141% zero entries)
Avg non-zero terms / doc = 166.2  (so với V = 193540)


In [3]:
terms = count_vec.get_feature_names_out()
df_ = np.bincount(C.indices, minlength=V)
idf_ = tfidf_tr.idf_
doc_id = 0
row = X[doc_id].toarray().ravel()
top_df = [(terms[i], int(df_[i])) for i in np.argsort(-df_)[:20]]
top_idf = [(terms[i], round(float(idf_[i]), 3)) for i in np.argsort(-idf_, kind="stable")[:20]]
top_tfidf = [(terms[i], round(float(row[i]), 3)) for i in np.argsort(-row)[:20] if row[i] > 0]
print(pd.DataFrame({"top DF (term, df)": top_df, "top IDF (term, idf)": top_idf,
                    f"top TF-IDF doc {doc_id} (term, w)": top_tfidf}).to_string())
print(f"\nDoc {doc_id} preview:", preview(docs[doc_id], 200))
print(f"#terms with df = 1 (max IDF): {(df_ == 1).sum()} / {V}")

   top DF (term, df)         top IDF (term, idf) top TF-IDF doc 0 (term, w)
0       (the, 27893)             (00000, 10.616)               (bbq, 0.462)
1       (and, 27423)            (000000, 10.616)             (class, 0.276)
2        (to, 26689)          (00000000, 10.616)              (meat, 0.194)
3        (of, 26031)  (0000000000000965, 10.616)             (balay, 0.179)
4        (in, 25224)          (00000001, 10.616)              (kcbs, 0.173)
5       (for, 23651)          (00000048, 10.616)          (lonestar, 0.168)
6        (is, 22739)            (000002, 10.616)              (will, 0.155)
7      (with, 21405)      (000004b926f1, 10.616)          (missoula, 0.151)
8        (on, 20262)          (00000781, 10.616)             (apron, 0.141)
9      (that, 18370)        (0000085054, 10.616)             (smoker, 0.14)
10     (this, 17840)          (00000xxx, 10.616)               (you, 0.137)
11      (are, 17594)            (000016, 10.616)         (timelines, 0.133)
12       (it

**✍️ TỰ VIẾT — Câu hỏi Part D**
- Tại sao một document chỉ dùng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều V? …
- So sánh 3 danh sách trên: …
- Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không? …
- Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không? …

## Part E — Core implementation
Xem `implementation.py` (không dùng `TfidfVectorizer` cho phần lõi). Chạy lại test tại đây:

In [4]:
import implementation
implementation.run_tests()
implementation.compare_with_sklearn()

  [PASS] test_tokenize
  [PASS] test_build_vocabulary
  [PASS] test_compute_counts
  [PASS] test_compute_tf
  [PASS] test_compute_df
  [PASS] test_compute_idf
  [PASS] test_compute_tfidf
  [PASS] test_cosine_similarity
  [PASS] test_tiny_tfidf_search
All 9 tests passed.

Vocabulary (ours)   : ['cat', 'dog', 'eats', 'fish', 'likes']
Vocabulary (sklearn): ['cat', 'dog', 'eats', 'fish', 'likes']

D1 ours    : [0.1352, 0.0, 0.1352, 0.0, 0.0]
D1 sklearn : [0.6198, 0.0, 0.6198, 0.4813, 0.0]

D1 ours, sklearn convention: [0.6198, 0.0, 0.6198, 0.4813, 0.0]
max |ours(sklearn convention) - sklearn| = 0.00e+00
-> The difference comes only from conventions (tf, idf smoothing, L2 norm).


## Part F — Experiment 2: Preprocessing ablation
- **A — Minimal:** lowercase → whitespace tokenization
- **B — Normalized:** lowercase → bỏ punctuation (giữ `[a-z0-9]+`) → tokenization → bỏ stopwords (sklearn English)
- **C — Extended:** Unicode normalization (NFKD, bỏ dấu, lowercase) → subword tokenization (WordPiece 30K, train trên chính corpus)

OOV rate: fit vocabulary trên 27,000 docs đầu, đo tỉ lệ token của 3,000 docs còn lại không có trong vocabulary.

In [5]:
t0 = time.time()
subword = train_subword_tokenizer(docs, vocab_size=30_000)
tokenize_c = make_tokenize_c(subword)
docs_c = pretokenize_c(subword, docs)
print(f"WordPiece trained + corpus encoded in {time.time()-t0:.0f}s")
print("Ví dụ:", tokenize_c("Myocardial infarction treatment with transformers"))

identity = lambda toks: toks
tok_docs = {"A": [tokenize_a(d) for d in docs],
            "B": [tokenize_b(d) for d in docs],
            "C": docs_c}
engines = {}
for name, toks in tok_docs.items():
    vec = TfidfVectorizer(analyzer=identity)
    Xp = vec.fit_transform(toks)
    prep = {"A": tokenize_a, "B": tokenize_b, "C": tokenize_c}[name]
    engines[name] = TfidfSearch(vec, Xp, prep=prep)

def oov_rate(toks, split=27_000):
    train_vocab = set(t for d in toks[:split] for t in d)
    held = [t for d in toks[split:] for t in d]
    return sum(t not in train_vocab for t in held) / len(held)

ablation = {}
for name, e in engines.items():
    Xp = e.X
    ablation[name] = {
        "Vocabulary size": Xp.shape[1],
        "Average tokens/document": round(np.mean([len(t) for t in tok_docs[name]]), 1),
        "Matrix sparsity": round(1 - Xp.nnz / (Xp.shape[0] * Xp.shape[1]), 6),
        "OOV rate (held-out 3K)": f"{oov_rate(tok_docs[name]) * 100:.2f}%",
    }
print(pd.DataFrame(ablation).rename(columns=lambda c: f"Pipeline {c}").to_string())

WordPiece trained + corpus encoded in 64s
Ví dụ: ['myocardial', 'inf', '##ar', '##ction', 'treatment', 'with', 'transformers']
                        Pipeline A Pipeline B Pipeline C
Vocabulary size             473388     186302      28553
Average tokens/document      361.1      197.5      459.0
Matrix sparsity           0.999611   0.999329   0.993311
OOV rate (held-out 3K)       3.98%      3.27%      0.13%


Ảnh hưởng của lowercasing (chỉ đổi 1 yếu tố): vocabulary khi **không** lowercase so với có lowercase, cùng whitespace tokenizer.

In [6]:
v_cased = len(set(t for d in docs for t in d.split()))
v_lower = len(set(t for d in tok_docs["A"] for t in d))
print(f"Whitespace, cased  : {v_cased}")
print(f"Whitespace, lower  : {v_lower}  (giảm {(1 - v_lower / v_cased) * 100:.1f}%)")
print("Ví dụ token 'image' trong Pipeline A:",
      sorted(t for t in engines["A"].vocab if t.startswith("image") and len(t) <= 7)[:12])

Whitespace, cased  : 542753
Whitespace, lower  : 473388  (giảm 12.8%)
Ví dụ token 'image' trong Pipeline A: ['image', 'image"', "image'.", "image's", 'image(', 'image)', 'image),', 'image).', 'image,', 'image-', 'image.', 'image..']


## Part G — Application: Document search
Query → TF-IDF query vector → cosine similarity (các hàng của X đã chuẩn hoá L2 nên `X @ q` = cosine) → top-K.

In [7]:
def show_search(query, engine, k=5):
    rows = [{"Rank": r, "Document ID": i, "Similarity": round(s, 4), "Document preview": preview(docs[i], 80)}
            for r, (i, s) in enumerate(engine.search(query, k), 1)]
    print(f"\nQuery: {query!r}")
    print(pd.DataFrame(rows).to_string(index=False))

for q in ["medical image classification", "transformer language model",
          "deep learning healthcare", "natural language processing"]:
    show_search(q, engines["B"])


Query: 'medical image classification'
 Rank  Document ID  Similarity                                                                 Document preview
    1        18971      0.4244 The new RTS Environmental Classification system (RTS GLT) is designed for partie
    2         8527      0.3537 History of maize classification. How races used in classification. Geographical 
    3        19908      0.2609 Download League Of Legends Wallpapers in high-quality for your desktop and smart
    4        17794      0.2596 Filters the output of 'wp_calculate_image_sizes()'. A source size value for use 
    5        12658      0.2490 This guidance is for pharmacists who handle, use and sell/supply medical devices

Query: 'transformer language model'
 Rank  Document ID  Similarity                                                                 Document preview
    1        27936      0.4794 hi, I am having problems with transformer / circuit board om my Hobby 720 uml mo
    2        25428      0.30

## Part H — Evaluation
**Evaluation set** (`eval_set.json`): 8 queries. Relevance labels được gán thủ công bằng *pooling*:
gộp top-10 của cả 3 pipeline cho mỗi query, đọc từng document và đánh dấu relevant nếu nội dung
thực sự nói về chủ đề của query (không chỉ chứa từ khoá).
Hai query không có document relevant nào trong pool được giữ lại để phân tích lỗi; Recall@5 của chúng không xác định (NaN) và được bỏ khỏi trung bình R@5.

In [8]:
eval_set = json.load(open("eval_set.json", encoding="utf-8"))

def precision_at_k(ret, rel, k=5):
    return sum(d in rel for d in ret[:k]) / k

def recall_at_k(ret, rel, k=5):
    return sum(d in rel for d in ret[:k]) / len(rel) if rel else float("nan")

def reciprocal_rank(ret, rel):
    for r, d in enumerate(ret, 1):
        if d in rel:
            return 1 / r
    return 0.0

K = 5
records, csv_rows = [], []
for name, e in engines.items():
    for qi, item in enumerate(eval_set, 1):
        rel = set(item["relevant"])
        res = e.search(item["query"], K)
        ret = [i for i, _ in res]
        p, r, rr = precision_at_k(ret, rel, K), recall_at_k(ret, rel, K), reciprocal_rank(ret, rel)
        records.append({"pipeline": name, "query": item["query"], "P@5": p, "R@5": r, "RR": rr})
        for rank, (d, s) in enumerate(res, 1):
            csv_rows.append({"pipeline": name, "query_id": qi, "query": item["query"], "rank": rank,
                             "doc_id": d, "similarity": round(s, 6), "is_relevant": d in rel,
                             "P@5": p, "R@5": r, "RR": round(rr, 4), "preview": preview(docs[d], 100)})
metrics = pd.DataFrame(records)
pd.DataFrame(csv_rows).to_csv("results.csv", index=False, encoding="utf-8")

print(metrics.pivot(index="query", columns="pipeline", values="P@5").round(2).to_string())
summary = metrics.groupby("pipeline")[["P@5", "R@5", "RR"]].mean().rename(columns={"RR": "MRR"}).round(3)
print("\nSearch performance (mean over queries):")
print(summary.T.to_string())
print("\nSaved results.csv with", len(csv_rows), "rows")

pipeline                        A    B    C
query                                      
deep learning healthcare      0.2  0.2  0.2
dog training tips             0.4  0.4  0.4
electric car battery          0.0  0.0  0.2
heart attack treatment        0.6  0.4  0.4
home mortgage interest rates  1.0  1.0  1.0
medical image classification  0.0  0.0  0.0
natural language processing   0.0  0.0  0.2
transformer language model    0.0  0.0  0.0

Search performance (mean over queries):
pipeline      A      B      C
P@5       0.275  0.250  0.300
R@5       0.509  0.454  0.620
MRR       0.354  0.406  0.479

Saved results.csv with 120 rows


**✍️ TỰ VIẾT — Câu hỏi phân tích 9.5** (trả lời bằng số liệu ở trên)
1. Lowercasing làm thay đổi vocabulary như thế nào? …
2. Stopword removal có luôn cải thiện representation không? …
3. Việc loại punctuation có thể làm mất thông tin gì? …
4. Pipeline nào tạo ra sparse matrix nhất? …
5. Pipeline nào cho search tốt nhất? …
6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không? …

## Part I — Error analysis (evidence)
Với mỗi query: top-5 của Pipeline B, các term đóng góp nhiều nhất vào cosine (`q_t · d_t`), và các relevant doc bị bỏ sót.

In [9]:
e = engines["B"]
for item in eval_set:
    q, rel = item["query"], set(item["relevant"])
    print("=" * 100)
    print("Query:", q, "| query tokens:", tokenize_b(q))
    print("Expected relevant:", sorted(rel) or "(none in pool)")
    res = e.search(q, 5)
    for r, (d, s) in enumerate(res, 1):
        mark = "✔" if d in rel else "✘"
        print(f" {r}. {mark} doc {d:<6} cos={s:.3f} top terms={e.contributions(q, d, 3)}  | {preview(docs[d], 70)}")
    missed = [d for d in rel if d not in {i for i, _ in res}]
    for d in missed:
        rank = int((e.scores(q) > e.scores(q)[d]).sum()) + 1
        print(f"    missed doc {d}: rank {rank}, overlap={e.contributions(q, d, 3)} | {preview(docs[d], 70)}")

Query: medical image classification | query tokens: ['medical', 'image', 'classification']
Expected relevant: (none in pool)
 1. ✘ doc 18971  cos=0.424 top terms=[('classification', 0.4244)]  | The new RTS Environmental Classification system (RTS GLT) is designed 
 2. ✘ doc 8527   cos=0.354 top terms=[('classification', 0.3537)]  | History of maize classification. How races used in classification. Geo
 3. ✘ doc 19908  cos=0.261 top terms=[('image', 0.2609)]  | Download League Of Legends Wallpapers in high-quality for your desktop
 4. ✘ doc 17794  cos=0.260 top terms=[('image', 0.2596)]  | Filters the output of 'wp_calculate_image_sizes()'. A source size valu
 5. ✘ doc 12658  cos=0.249 top terms=[('medical', 0.249)]  | This guidance is for pharmacists who handle, use and sell/supply medic
Query: transformer language model | query tokens: ['transformer', 'language', 'model']
Expected relevant: (none in pool)
 1. ✘ doc 27936  cos=0.479 top terms=[('transformer', 0.4231), ('model', 0.0563)

Failure case: *"heart attack treatment"* vs *"myocardial infarction therapy"* — không có lexical overlap.

In [10]:
pair = ["heart attack treatment", "myocardial infarction therapy"]
for name, e in engines.items():
    qv = e.vectorizer.transform([e.prep(pair[0])])
    dv = e.vectorizer.transform([e.prep(pair[1])])
    cos = (qv @ dv.T).toarray()[0, 0]
    print(f"Pipeline {name}: cos = {cos:.4f}   tokens: {e.prep(pair[0])} | {e.prep(pair[1])}")
hits = [i for i, d in enumerate(docs) if "myocardial infarction" in d.lower()]
print("\nDocs containing 'myocardial infarction':", hits[:10])
for d in hits[:5]:
    rank = int((engines["B"].scores(pair[0]) > engines["B"].scores(pair[0])[d]).sum()) + 1
    print(f"  doc {d}: rank {rank} for query {pair[0]!r} | {preview(docs[d], 80)}")

Pipeline A: cos = 0.0000   tokens: ['heart', 'attack', 'treatment'] | ['myocardial', 'infarction', 'therapy']
Pipeline B: cos = 0.0000   tokens: ['heart', 'attack', 'treatment'] | ['myocardial', 'infarction', 'therapy']
Pipeline C: cos = 0.0000   tokens: ['heart', 'attack', 'treatment'] | ['myocardial', 'inf', '##ar', '##ction', 'therapy']

Docs containing 'myocardial infarction': [1422, 5052, 8343, 9884, 12163, 12817, 16842, 19669, 20893, 26337]
  doc 1422: rank 2 for query 'heart attack treatment' | I. Causes of Heart Failure: What every physician needs to know. Heart failure (H
  doc 5052: rank 450 for query 'heart attack treatment' | Mechanosensitive Ion Channels in Cardiovascular Physiology. Exp Clin Cardiol 201
  doc 8343: rank 2345 for query 'heart attack treatment' | Background: Statin therapy is associated with improved survival in patients at h
  doc 9884: rank 2345 for query 'heart attack treatment' | Background: The trials comparing Minimally Invasive Direct Coronary Artery

**✍️ TỰ VIẾT — Error analysis**: chọn 2 query tốt + 2 query kém ở trên, trả lời 5 câu hỏi của Part I
(vì sao doc đứng đầu, từ nào đóng góp, lexical overlap, doc bị bỏ sót, failure xuất phát từ đâu),
và giải thích failure case quan trọng nhất.

## Part J — From failure to the next representation
**✍️ TỰ VIẾT — Hypothesis**: …